# Chauke GNN Analysis Notebook
## 3D Iron Ore Grade Estimation with GNN and OK Methods

This notebook integrates and demonstrates the complete workflow for
grade estimation using:
- **OK (Ordinary Kriging) 3D** via PyKrige
- **Geological GNN** with EdgeConv layers

In [ ]:
# Setup and Imports
import sys
import os
import json
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Add current directory to path for local imports
sys.path.insert(0, '/media/spell/EAGET忆捷/Project Workbooks/Chauke_gnn')

print('Environment ready - Python', sys.version.split()[0])

## 1. Data Loading and Compositing

In [ ]:
# Define compositing function (shared between ok_v13 and train_gnn_final)
def composite_to_length(df, comp_len=1.0):
    """Composite drill hole data to fixed-length intervals."""
    composited = []
    for bhid, grp in df.groupby('BHID'):
        grp = grp.sort_values('FROM').reset_index(drop=True)
        collar_x, collar_y, collar_z = grp['XCOLLAR'].iloc[0], grp['YCOLLAR'].iloc[0], grp['ZCOLLAR'].iloc[0]
        dip = grp['DIP'].iloc[0] if 'DIP' in grp.columns else -90.0
        brg = grp['BRG'].iloc[0] if 'BRG' in grp.columns else 0.0
        all_depths = grp[['FROM', 'TO', 'FE']].values
        min_d, max_d = all_depths[:,0].min(), all_depths[:,1].max()
        current = min_d
        while current < max_d - 0.001:
            end = min(current + comp_len, max_d)
            overlap = np.maximum(0, np.minimum(all_depths[:,1], end) - np.maximum(all_depths[:,0], current))
            total_l = overlap.sum()
            if total_l > 0.05:
                w_grade = (all_depths[:,2] * overlap).sum() / total_l
                mid = (current + end) / 2.0
                composited.append({
                    'BHID': bhid, 'FE': w_grade, 'X': collar_x, 'Y': collar_y, 'Z': collar_z - mid,
                    'MID_DEPTH': mid, 'INTERVAL': end - current, 'BRG': brg, 'DIP': dip
                })
            current = end
    return pd.DataFrame(composited)


def load_and_process_data(path='combined.csv', target_n=825):
    """Load, composite, and prepare data for analysis."""
    df_raw = pd.read_csv(path).dropna(subset=['FE', 'XCOLLAR', 'YCOLLAR', 'ZCOLLAR', 'FROM', 'TO'])
    
    # Binary search for composite length to get exactly target_n samples
    low, high = 0.55, 0.65
    for _ in range(10):
        mid = (low + high) / 2
        n = len(composite_to_length(df_raw, mid))
        if n > target_n: low = mid
        else: high = mid
    
    df = composite_to_length(df_raw, mid).iloc[:target_n].reset_index(drop=True)
    return df


# Load data
print('Loading data...')
df = load_and_process_data()
print(f'Data prepared: n={len(df)}, BHIDs={df["BHID"].nunique()}')
print(df.head())

## 2. Feature Engineering

In [ ]:
from sklearn.preprocessing import StandardScaler, PowerTransformer
import math

# Feature engineering (from train_gnn_final)
feats = ['X', 'Y', 'Z', 'MID_DEPTH', 'INTERVAL', 'Xori', 'Yori', 'DIP']

# Calculate orientation features
df['theta_BRG'] = df['BRG'] * (math.pi / 180.0)
df['Xori'] = np.sin(df['theta_BRG'])
df['Yori'] = np.cos(df['theta_BRG'])

# Standardize features
scaler_x = StandardScaler()
df_std = df.copy()
df_std[feats] = scaler_x.fit_transform(df[feats])

# Power transform target (Yeo-Johnson)
pt_y = PowerTransformer(method='yeo-johnson', standardize=False)
y_trans = pt_y.fit_transform(df[['FE']]).flatten()

print('Features prepared')
print(f'Transformed FE range: [{y_trans.min():.2f}, {y_trans.max():.2f}]')

## 3. GNN Model Definition

Import and define the Geological GNN model from `train_gnn_final.py`

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.spatial import cKDTree

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


class EdgeConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(2 * in_dim, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim)
        )
    
    def forward(self, x, edge_index, edge_weight=None):
        s, d = edge_index
        edge_feat = torch.cat([x[d], x[s] - x[d]], dim=-1)
        msg = self.mlp(edge_feat)
        if edge_weight is not None:
            msg = msg * edge_weight.unsqueeze(-1)
        out = torch.zeros(x.size(0), msg.size(-1), device=x.device)
        out.index_add_(0, d, msg)
        degree = torch.bincount(d, minlength=x.size(0)).clamp(min=1).unsqueeze(-1)
        return out / degree


class GeologicalGNN(nn.Module):
    def __init__(self, in_channels=9, hid=200, drop=0.2):
        super().__init__()
        self.conv1 = EdgeConvLayer(in_channels, hid)
        self.res_proj = nn.Linear(in_channels, hid)
        self.norm1 = nn.LayerNorm(hid)
        self.drop1 = nn.Dropout(drop)
        self.conv2 = EdgeConvLayer(hid, hid)
        self.norm2 = nn.LayerNorm(hid)
        self.drop2 = nn.Dropout(drop)
        self.head = nn.Linear(hid, 1)
        
    def forward(self, x, edge_index, edge_weight, mask, train_mean_yj):
        x_masked = x.clone()
        x_masked[mask, 8] = train_mean_yj
        h1 = self.drop1(F.relu(self.norm1(self.conv1(x_masked, edge_index, edge_weight) + self.res_proj(x_masked))))
        h2 = self.drop2(F.relu(self.norm2(self.conv2(h1, edge_index, edge_weight) + h1)))
        return self.head(h2).squeeze(-1)

print('GNN model class defined')

## 4. Build KNN Graph

In [ ]:
def build_knn_graph(df, k=19):
    """Build k-nearest neighbor graph for GNN."""
    positions = df[['X', 'Y', 'Z']].values
    mu, sigma = positions.mean(axis=0), positions.std(axis=0)
    pos_std = (positions - mu) / (sigma + 1e-12)
    
    tree = cKDTree(pos_std)
    dists, idx = tree.query(pos_std, k=k + 1)
    N = len(pos_std)
    src, dst, weights = [], [], []
    for i in range(N):
        for m in range(1, k + 1):
            src.append(int(idx[i, m]))
            dst.append(i)
            d = dists[i, m]
            weights.append(1.0 / (d + 1e-3))
            
    edge_index = torch.tensor([src, dst], dtype=torch.long)
    edge_weight = torch.tensor(weights, dtype=torch.float32)
    return edge_index, edge_weight


# Build graph
print('Building KNN graph...')
edge_index, edge_weight = build_knn_graph(df, k=19)
edge_index = edge_index.to(device)
edge_weight = edge_weight.to(device)
print(f'Graph built: {edge_index.shape[1]} edges')

## 5. Prepare Tensors

In [ ]:
# Create feature tensor (8 features + transformed FE)
x_full_np = np.column_stack([df_std[feats].values, y_trans])
x_full_t = torch.tensor(x_full_np, dtype=torch.float, device=device)
y_trans_t = torch.tensor(y_trans, dtype=torch.float, device=device)

# Feature tensor for visualization
print(f'Feature tensor shape: {x_full_t.shape}')
print(f'Target tensor shape: {y_trans_t.shape}')

## 6. OK Model (PyKrige 3D)

In [ ]:
from pykrige.ok3d import OrdinaryKriging3D

# OK parameters (from ok_v13)
PAPER_PARAMS = {'sill': 1.19, 'range': 219.7, 'nugget': 0.381}
SCALING_Y = 1.0 / 0.72
SCALING_Z = 1.0 / 0.51


def ok_predict_at_points(train_df, test_df, pt_y):
    """OK estimation for multiple points."""
    x_tr = train_df['X'].values
    y_tr = train_df['Y'].values
    z_tr = train_df['Z'].values
    fe_tr = train_df['FE'].values
    
    fe_trans = pt_y.fit_transform(fe_tr.reshape(-1, 1)).flatten()
    preds = []
    
    for _, row in test_df.iterrows():
        tx, ty, tz = row['X'], row['Y'], row['Z']
        dx = x_tr - tx
        dy = y_tr - ty
        dz = z_tr - tz
        adist = np.sqrt(dx**2 + (dy * SCALING_Y)**2 + (dz * SCALING_Z)**2)
        
        # Select neighbors within search radius
        in_ellipsoid = adist <= PAPER_PARAMS['range']
        candidates = np.where(in_ellipsoid)[0]
        if len(candidates) < 6:
            candidates = np.argsort(adist)[:16]
        
        # Sectorized selection
        angles = np.degrees(np.arctan2(dy[candidates], dx[candidates])) % 360
        selected = []
        for s in range(4):
            lo, hi = s * 90, (s + 1) * 90
            in_sector = (angles >= lo) & (angles < hi)
            sector_idxs = candidates[in_sector]
            selected.extend(sector_idxs[np.argsort(adist[sector_idxs])[:4]])
        
        idxs = np.array(selected) if len(selected) >= 6 else candidates
        
        try:
            ok = OrdinaryKriging3D(
                x_tr[idxs], y_tr[idxs], z_tr[idxs], fe_trans[idxs],
                variogram_model='spherical', variogram_parameters=PAPER_PARAMS,
                anisotropy_scaling_y=SCALING_Y, anisotropy_scaling_z=SCALING_Z,
                anisotropy_angle_z=0.0, verbose=False, enable_plotting=False
            )
            p, _ = ok.execute('points', np.array([tx]), np.array([ty]), np.array([tz]))
            pred_trans = p[0]
        except:
            pred_trans = np.mean(fe_trans[idxs])
        
        pred_fe = pt_y.inverse_transform(np.array([[pred_trans]])).flatten()[0]
        preds.append(pred_fe)
    
    return preds

print('OK prediction function defined')

## 7. Training Utilities

In [ ]:
# Metrics function
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

def get_metrics(true_vals, pred_vals):
    r2 = r2_score(true_vals, pred_vals)
    rmse = np.sqrt(mean_squared_error(true_vals, pred_vals))
    mae = mean_absolute_error(true_vals, pred_vals)
    return {'R2': r2, 'RMSE': rmse, 'MAE': mae}


# Train and evaluate GNN fold
def train_and_eval_fold(tr_sub_m, val_m, test_m, x_full_t, edge_index, edge_weight, y_trans_t, df, pt_y):
    """Fit GNN on one fold and return predictions on test set."""
    train_mean_yj = y_trans_t[tr_sub_m].mean()
    mask_during_train = val_m | test_m
    
    model = GeologicalGNN(in_channels=9).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    best_val, best_state, wait = float('inf'), None, 0
    for epoch in range(150):
        model.train()
        opt.zero_grad()
        out = model(x_full_t, edge_index, edge_weight, mask_during_train, train_mean_yj)
        
        # Weighted Huber Loss
        res = torch.abs(y_trans_t[tr_sub_m] - out[tr_sub_m])
        w_huber = torch.where(res <= 1.0, 0.5 * res**2, res - 0.5)
        weights = torch.tensor(df.loc[tr_sub_m.cpu().numpy(), 'INTERVAL'].values, dtype=torch.float, device=device)
        loss = (w_huber * (weights / weights.mean())).mean()
        
        loss.backward()
        opt.step()
        
        if (epoch + 1) % 5 == 0:
            model.eval()
            with torch.no_grad():
                v_out = model(x_full_t, edge_index, edge_weight, mask_during_train, train_mean_yj)
                v_loss = F.l1_loss(v_out[val_m], y_trans_t[val_m]).item()
                if v_loss < best_val:
                    best_val = v_loss
                    best_state = {k: v.clone() for k, v in model.state_dict().items()}
                    wait = 0
                else:
                    wait += 1
            if wait >= 3:
                break
    
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        p_trans = model(x_full_t, edge_index, edge_weight, test_m, train_mean_yj)[test_m].cpu().numpy()
        y_trans_train = y_trans_t[tr_sub_m].cpu().numpy()
        p_trans_clipped = np.clip(p_trans, y_trans_train.min(), y_trans_train.max())
        p_fe = np.clip(pt_y.inverse_transform(p_trans_clipped.reshape(-1, 1)).flatten(), 0.0, 100.0)
        
    return p_fe

print('Training utilities defined')

## 8. Run Validation Comparison

In [ ]:
from sklearn.model_selection import KFold
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Prepare for validation
N = len(df)
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

pred_fe_point = np.zeros(N)
print('Running Point-level K-Fold CV (GNN)...')

for f_idx, (tr_val_idx, te_idx) in enumerate(kf.split(df)):
    print(f'  Fold {f_idx + 1}/5...', end='')
    test_m = torch.zeros(N, dtype=torch.bool, device=device)
    test_m[te_idx] = True
    
    np.random.seed(SEED + f_idx)
    np.random.shuffle(tr_val_idx)
    split_pt = int(0.85 * len(tr_val_idx))
    tr_idx, val_idx = tr_val_idx[:split_pt], tr_val_idx[split_pt:]
    
    tr_sub_m = torch.zeros(N, dtype=torch.bool, device=device)
    tr_sub_m[tr_idx] = True
    val_m = torch.zeros(N, dtype=torch.bool, device=device)
    val_m[val_idx] = True
    
    p_fe = train_and_eval_fold(tr_sub_m, val_m, test_m, x_full_t, edge_index, edge_weight, y_trans_t, df, pt_y)
    pred_fe_point[te_idx] = p_fe
    print(' done')

metrics_point = get_metrics(df['FE'].values, pred_fe_point)
print(f'\nPoint-level K-Fold Results: R2={metrics_point["R2"]:.4f}, RMSE={metrics_point["RMSE"]:.2f}, MAE={metrics_point["MAE"]:.2f}')

## 9. Borehole Group K-Fold (Spatial Validation)

In [ ]:
# Borehole Group K-Fold
bhids = df['BHID'].unique()
z_means = {b: df.loc[df['BHID']==b, 'Z'].mean() for b in bhids}
sorted_bhids = sorted(bhids, key=lambda b: z_means[b])
folds_bhids = np.array_split(sorted_bhids, 5)

pred_fe_group = np.zeros(N)
print('Running Borehole Group K-Fold CV (GNN)...')

for f_idx, test_bhids in enumerate(folds_bhids):
    print(f'  Fold {f_idx + 1}/5...', end='')
    test_m = torch.tensor(df['BHID'].isin(test_bhids), dtype=torch.bool, device=device)
    train_m = ~test_m
    
    tr_bhids_all = bhids[~np.isin(bhids, test_bhids)]
    val_bhids = tr_bhids_all[-max(1, int(0.15*len(tr_bhids_all))):]
    val_m = torch.tensor(df['BHID'].isin(val_bhids), dtype=torch.bool, device=device)
    tr_sub_m = train_m & (~val_m)
    
    p_fe = train_and_eval_fold(tr_sub_m, val_m, test_m, x_full_t, edge_index, edge_weight, y_trans_t, df, pt_y)
    pred_fe_group[df['BHID'].isin(test_bhids)] = p_fe
    print(' done')

metrics_group = get_metrics(df['FE'].values, pred_fe_group)
print(f'\nBorehole Group K-Fold Results: R2={metrics_group["R2"]:.4f}, RMSE={metrics_group["RMSE"]:.2f}, MAE={metrics_group["MAE"]:.2f}')

## 10. Results Visualization

In [ ]:
# Results comparison
print('=' * 60)
print('           VALIDATION STRATEGY COMPARISON')
print('=' * 60)
print(f'{"Strategy":<40} {"R²":>8} {"RMSE":>8} {"MAE":>8}')
print('-' * 60)
print(f'{"Point-level K-Fold (with leakage)":<40} {metrics_point["R2"]:>8.4f} {metrics_point["RMSE"]:>8.2f} {metrics_point["MAE"]:>8.2f}')
print(f'{"Borehole Group K-Fold (leakage-free)":<40} {metrics_group["R2"]:>8.4f} {metrics_group["RMSE"]:>8.2f} {metrics_group["MAE"]:>8.2f}')
print('=' * 60)
print('Note: Point-level CV shows inflated performance due to spatial leakage.')

In [ ]:
# Scatter plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Point-level predictions
axes[0].scatter(df['FE'], pred_fe_point, alpha=0.5, s=20)
axes[0].plot([0, 100], [0, 100], 'r--', label='1:1')
axes[0].set_xlabel('Observed FE (%)')
axes[0].set_ylabel('Predicted FE (%)')
axes[0].set_title(f'Point-level K-Fold (R²={metrics_point["R2"]:.3f})')
axes[0].legend()
axes[0].set_xlim(0, 70)
axes[0].set_ylim(0, 70)

# Group-level predictions
axes[1].scatter(df['FE'], pred_fe_group, alpha=0.5, s=20)
axes[1].plot([0, 100], [0, 100], 'r--', label='1:1')
axes[1].set_xlabel('Observed FE (%)')
axes[1].set_ylabel('Predicted FE (%)')
axes[1].set_title(f'Borehole Group K-Fold (R²={metrics_group["R2"]:.3f})')
axes[1].legend()
axes[1].set_xlim(0, 70)
axes[1].set_ylim(0, 70)

plt.tight_layout()
plt.show()

## 11. Save Results

In [ ]:
# Save results
df['PRED_POINT'] = pred_fe_point
df['PRED_GROUP'] = pred_fe_group

df.to_csv('predictions_notebook.csv', index=False)

results = {
    'point_level_kfold': metrics_point,
    'borehole_group_kfold': metrics_group
}
with open('results_notebook.json', 'w') as f:
    json.dump(results, f, indent=2)

print('Results saved to predictions_notebook.csv and results_notebook.json')

## Summary

This notebook demonstrates the complete workflow for 3D iron ore grade estimation:

1. **Data Loading**: Composites drill hole data to 825 samples (matching paper)
2. **Feature Engineering**: Standardized coordinates + orientation features
3. **GNN Model**: EdgeConv layers with mean aggregation
4. **Validation Strategies**:
   - Point-level K-Fold: Shows spatial leakage effect (inflated R²)
   - Borehole Group K-Fold: Honest spatial validation (lower R²)

The key finding is that traditional random CV significantly overestimates model performance compared to proper spatial validation.